# Continue CPI training to 60 total epochs
Resume the **existing Davis, KIBA and Metz checkpoints**, with validation-only comparison and recovery ZIPs. This does not train protein structures or physical binding motion.

1. Import this notebook into Kaggle. Attach the original **cpi_kaggle.zip** training dataset and your **cpi_training_checkpoints.zip** as private inputs. Kaggle may extract the ZIPs automatically; both forms are supported.
2. Select **GPU T4 x2**, enable Internet for dependencies, then run all cells in order.
3. Keep batch size 16. Target 60 means 30 more epochs for the supplied 30-epoch checkpoints. To extend only Davis, set `DATASETS_TO_TRAIN = ('davis',)`.
4. Download both recovery ZIPs and the validation report before ending the session. Keep your original input ZIP separately.

The saved baseline best validation MSEs are Davis **0.329098** (epoch 30), KIBA **0.269671** (28), Metz **0.458601** (25). Lower MSE is better. These are separate dataset scales; do not compare them as dataset rankings. More epochs may not improve them.

Only load your own trusted checkpoint files: PyTorch checkpoints can contain executable Python objects.


In [ ]:
from pathlib import Path
ROOT = Path('/kaggle/working/cpi_resume60')
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working')
TARGET_EPOCHS = 60
BATCH_SIZE = 16
DATASETS_TO_TRAIN = ('davis', 'kiba', 'metz')
RUN_CONTINUATION = True
SNAPSHOT_INTERVAL_SECONDS = 600  # A ZIP every 10 minutes, plus before/after training.
TRAINING_PACKAGE = None  # Set exact ZIP or extracted package folder if auto-detection is ambiguous.
RESTORE_ZIP = None       # Set exact checkpoint ZIP or extracted training_runs folder if needed.


## Recovery helpers
This cell only defines functions. Inputs are copied into a separate working folder. Re-running setup reuses its saved progress; it does not restore old checkpoints over newer ones. The checkpoint, original split hashes, batch size, architecture and seed must match before training.


In [ ]:
"""CPI continuation helpers. Embedded in CPI_T4x2_resume_60.ipynb."""
import copy
import hashlib
import io
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import tempfile
import threading
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

DATASETS = ('davis', 'kiba', 'metz')


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def extract_checked(archive, destination, prefix=None):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as source:
        for member in source.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination) or '\\' in member.filename:
                raise ValueError('Unsafe ZIP path; the archive was not restored.')
            if prefix and not member.filename.startswith(prefix):
                raise ValueError('Checkpoint ZIP must contain only training_runs/ files.')
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Symbolic links are not supported in the input ZIP.')
        source.extractall(destination)


def discover(inputs, archive_name, kind):
    """Find a single ZIP or Kaggle's already-extracted equivalent."""
    inputs = Path(inputs)
    archives = sorted(inputs.rglob(archive_name))
    if len(archives) == 1:
        return archives[0]
    if len(archives) > 1:
        raise ValueError(f'Multiple {kind} ZIPs found. Set its path explicitly in the first cell.')
    if kind == 'training package':
        folders = {p.parent.parent for p in inputs.rglob('train_cpi.py')
                   if p.parent.name == 'scripts' and (p.parent.parent / 'model_source/chemprop').is_dir()}
    else:
        folders = {p.parent.parent for p in inputs.rglob('run_manifest.json')
                   if p.parent.name == 'davis' and
                   all((p.parent.parent / ds / 'fold_0/model_0/resume.pt').is_file() for ds in DATASETS)}
    if len(folders) != 1:
        raise ValueError(f'Attach exactly one {kind}, or set its path explicitly. Nothing will train from scratch.')
    return next(iter(folders))


def prepare_root(root, inputs, package=None, restore=None):
    root = Path(root).resolve()
    marker = root / 'continuation_workspace.json'
    if marker.exists():
        if json.loads(marker.read_text())['schema'] != 'cpi-continuation-v1':
            raise ValueError('Unrecognized existing workspace. Choose a new ROOT.')
        print('Using existing continuation files; not restoring over them:', root)
        return root
    if root.exists() and any(root.iterdir()):
        raise ValueError('ROOT already contains other files. Choose a new continuation folder; it will not be overwritten.')
    package = Path(package) if package else discover(inputs, 'cpi_kaggle.zip', 'training package')
    restore = Path(restore) if restore else discover(inputs, 'cpi_training_checkpoints*.zip', 'checkpoint bundle')
    root.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='cpi-restore-', dir=root.parent) as temporary:
        stage = Path(temporary)
        if package.is_file():
            extract_checked(package, stage)
        else:
            for name in ('scripts', 'model_source', 'processed'):
                shutil.copytree(package / name, stage / name)
            shutil.copy2(package / 'requirements-kaggle.txt', stage / 'requirements-kaggle.txt')
        if (stage / 'training_runs').exists():
            raise ValueError('The training package must not contain checkpoints. Attach the original cpi_kaggle.zip.')
        if restore.is_file():
            extract_checked(restore, stage, 'training_runs/')
        else:
            runs = restore if (restore / 'davis/run_manifest.json').exists() else restore / 'training_runs'
            shutil.copytree(runs, stage / 'training_runs')
        for ds in DATASETS:
            for suffix in ('run_manifest.json', 'fold_0/model_0/model.pt', 'fold_0/model_0/resume.pt'):
                if not (stage / 'training_runs' / ds / suffix).is_file():
                    raise ValueError(f'{ds}: missing {suffix}. Attach the complete checkpoint bundle.')
        (stage / 'continuation_workspace.json').write_text(json.dumps(dict(
            schema='cpi-continuation-v1', package=str(package), checkpoint_source=str(restore),
            package_sha256=sha256(package) if package.is_file() else None,
            checkpoint_sha256=sha256(restore) if restore.is_file() else None), indent=2))
        if root.exists():
            root.rmdir()  # Only the empty folder verified above.
        stage.rename(root)
    return root


class Continuation:
    def __init__(self, root, output_dir, target=60, selected=DATASETS, batch_size=16):
        import torch
        self.torch = torch
        self.root, self.output = Path(root), Path(output_dir)
        self.target, self.selected, self.batch_size = target, tuple(selected), batch_size
        if type(target) is not int or target < 31 or batch_size != 16:
            raise ValueError('Use a total target above 30 and keep BATCH_SIZE=16 for these checkpoints.')
        if not self.selected or not set(self.selected) <= set(DATASETS) or len(set(self.selected)) != len(self.selected):
            raise ValueError('Select Davis, KIBA and/or Metz once each.')
        self.manifests, self.templates, self.baseline = {}, {}, {}
        self.stop = threading.Event()
        self.output.mkdir(parents=True, exist_ok=True)
        self.preflight()

    def folder(self, dataset):
        return self.root / 'training_runs' / dataset

    def state(self, dataset):
        # The original trainer atomically replaces resume.pt after each epoch.
        # Reading through one open file captures that complete saved state.
        raw = (self.folder(dataset) / 'fold_0/model_0/resume.pt').read_bytes()
        state = self.torch.load(io.BytesIO(raw), map_location='cpu', weights_only=False)
        return raw, state

    def preflight(self):
        for ds in DATASETS:
            folder = self.folder(ds)
            manifest = json.loads((folder / 'run_manifest.json').read_text())
            expected = dict(dataset=ds, source_sha256={split: sha256(self.root / f'processed/{ds}/{split}.csv')
                            for split in ('train', 'val', 'test')}, batch_size=16, smoke=False, seed=42,
                            architecture='Recovered PerceiverCPI, sequence_length=500, original featurization')
            if manifest['signature'] != expected:
                raise ValueError(f'{ds}: data, batch size, seed or architecture differs. Nothing has been trained.')
            _, state = self.state(ds)
            required = {'epoch', 'model', 'optimizer', 'scheduler', 'best_model', 'best_score', 'best_epoch', 'rng', 'n_iter'}
            if not required <= state.keys() or state['best_model'] is None or int(state['epoch']) < 29:
                raise ValueError(f'{ds}: a complete checkpoint with at least 30 finished epochs is required.')
            if not math.isfinite(float(state['best_score'])):
                raise ValueError(f'{ds}: saved validation MSE is invalid.')
            self.manifests[ds] = manifest
            template_path = self.root / 'checkpoint_templates' / (ds + '.pt')
            if not template_path.exists():
                template = self.torch.load(folder / 'fold_0/model_0/model.pt', map_location='cpu', weights_only=False)
                template_path.parent.mkdir(exist_ok=True)
                self.torch.save(template, template_path.with_suffix('.tmp'))
                os.replace(template_path.with_suffix('.tmp'), template_path)
            self.templates[ds] = self.torch.load(template_path, map_location='cpu', weights_only=False)
            expected_weights = self.templates[ds]['state_dict']
            for name in ('model', 'best_model'):
                weights = state[name]
                if weights.keys() != expected_weights.keys() or any(weights[k].shape != expected_weights[k].shape for k in weights):
                    raise ValueError(f'{ds}: checkpoint architecture does not match the saved model.')
                if any(not self.torch.isfinite(t).all().item() for t in weights.values()):
                    raise ValueError(f'{ds}: {name} contains invalid numbers. Restore a known good checkpoint.')
            self.baseline[ds] = dict(completed_epochs=int(state['epoch'])+1,
                                     best_epoch=int(state['best_epoch'])+1,
                                     validation_mse=float(state['best_score']),
                                     learning_rates=[float(group['lr']) for group in state['optimizer']['param_groups']])
            print(ds, self.baseline[ds], flush=True)
        saved = self.root / 'continuation_baseline.json'
        if saved.exists():
            self.baseline = json.loads(saved.read_text())
        else:
            metadata = self.root / 'training_runs/continuation_snapshot.json'
            if metadata.exists():
                self.baseline = json.loads(metadata.read_text())['original_baseline']
            saved.write_text(json.dumps(self.baseline, indent=2))
        print('Preflight passed. Optimizer, RNG and original learning-rate state will be preserved.')
        print('The supplied train loop does not call scheduler.step(); no new learning-rate schedule is introduced.')

    def snapshot(self, baseline=False):
        name = 'cpi_training_checkpoints_baseline.zip' if baseline else f'cpi_training_checkpoints_resume_{self.target}.zip'
        destination = self.output / name
        if baseline and destination.exists():
            with zipfile.ZipFile(destination) as z:
                metadata = json.loads(z.read('training_runs/continuation_snapshot.json'))
            if metadata['original_baseline'] != self.baseline:
                raise ValueError('A different baseline ZIP exists. Choose a new OUTPUT directory.')
            print('Keeping the existing baseline archive:', destination)
            return destination
        previous_epochs = {}
        if destination.exists():
            with zipfile.ZipFile(destination) as z:
                previous_epochs = json.loads(z.read('training_runs/continuation_snapshot.json'))['completed_epochs']
        temporary = destination.with_suffix('.zip.tmp')
        epochs = {}
        with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
            for ds in DATASETS:
                folder = self.folder(ds)
                raw, state = self.state(ds)
                prefix = f'training_runs/{ds}/'
                z.writestr(prefix + 'fold_0/model_0/resume.pt', raw)
                # model.pt can be written concurrently. Rebuild the compatible
                # best checkpoint from the authoritative atomic resume state.
                best = dict(self.templates[ds])
                best['args'] = copy.copy(best['args'])
                best['args'].epochs = int(state['epoch']) + 1
                best['state_dict'] = state['best_model']
                buffer = io.BytesIO(); self.torch.save(best, buffer)
                z.writestr(prefix + 'fold_0/model_0/model.pt', buffer.getvalue())
                manifest = copy.deepcopy(self.manifests[ds])
                manifest['epochs'] = int(state['epoch']) + 1
                z.writestr(prefix + 'run_manifest.json', json.dumps(manifest, indent=2))
                for path in folder.rglob('*'):
                    if not path.is_file() or path.name in ('resume.pt', 'model.pt', 'run_manifest.json') or path.name.endswith('.tmp'):
                        continue
                    z.write(path, prefix + path.relative_to(folder).as_posix())
                epochs[ds] = int(state['epoch']) + 1
            z.writestr('training_runs/continuation_snapshot.json', json.dumps(dict(
                completed_epochs=epochs, original_baseline=self.baseline, target=self.target,
                note='resume.pt and rebuilt model.pt are authoritative. Logs may end mid-line. Training reports and test predictions may describe an earlier completed stage; read epochs_requested in each report.',
                created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())), indent=2))
        if any(epochs[ds] < previous_epochs.get(ds, 0) for ds in DATASETS):
            raise ValueError('Existing recovery ZIP contains newer progress. Restore that ZIP or choose a new OUTPUT folder; it was not overwritten.')
        os.replace(temporary, destination)
        print('Saved recovery ZIP:', destination, 'completed epochs:', epochs, flush=True)
        return destination

    def estimates(self):
        result = {}
        for ds in DATASETS:
            timings = []
            timing_file = self.folder(ds) / 'timings.jsonl'
            for line in timing_file.read_text().splitlines() if timing_file.exists() else []:
                try: timings.append(json.loads(line))
                except json.JSONDecodeError: pass
            training = [r['seconds'] for r in timings if r['phase'] == 'training_epoch'][-10:]
            validation = [r['seconds'] for r in timings if r['phase'] == 'validation_epoch'][-10:]
            _, state = self.state(ds)
            seconds = statistics.mean(training) + statistics.mean(validation) if training and validation else None
            remaining = max(0, self.target-int(state['epoch'])-1) if ds in self.selected else 0
            result[ds] = dict(epoch_minutes=seconds/60 if seconds else None, remaining_epochs=remaining,
                              estimated_hours=remaining*seconds/3600 if seconds else (0 if remaining == 0 else None))
        if any(r['estimated_hours'] is None for r in result.values()):
            print('Timing logs incomplete; a reliable estimate is unavailable.')
            return result
        wall = max(result['kiba']['estimated_hours'], result['davis']['estimated_hours']+result['metz']['estimated_hours'])
        print(json.dumps(result, indent=2))
        print(f'Estimated remaining wall time: {wall:.2f} hours plus overhead; not a guarantee.')
        return result

    def run_dataset(self, ds, gpu):
        if ds not in self.selected or self.stop.is_set():
            return
        _, state = self.state(ds)
        if int(state['epoch']) + 1 >= self.target:
            print(ds, 'already reached the total target; skipping.', flush=True)
            return
        command = [sys.executable, '-u', str(self.root / 'scripts/train_cpi.py'),
                   '--dataset', ds, '--epochs', str(self.target), '--batch-size', '16',
                   '--threads', '2', '--device', 'cuda', '--resume', '--output', str(self.folder(ds))]
        logs = self.root / 'kaggle_logs'; logs.mkdir(exist_ok=True)
        log = logs / f'{ds}_resume_{self.target}.log'
        print(f'{ds}: continue after {int(state["epoch"])+1} completed epochs on GPU {gpu}. Log: {log}', flush=True)
        del state
        with log.open('a') as stream:
            process = subprocess.Popen(command, cwd=self.root,
                env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1'), stdout=stream, stderr=subprocess.STDOUT)
            try:
                while process.poll() is None:
                    if self.stop.wait(1):
                        process.terminate()
                        try: process.wait(timeout=15)
                        except subprocess.TimeoutExpired: process.kill(); process.wait()
                        raise RuntimeError(f'{ds} stopped. Resume again from the last completed epoch.')
                if process.returncode:
                    raise RuntimeError(f'{ds} failed. Last log lines:\n' + log.read_text(errors='replace')[-4000:])
            finally:
                if process.poll() is None:
                    process.terminate()
                    try: process.wait(timeout=15)
                    except subprocess.TimeoutExpired: process.kill(); process.wait()
        print(ds, 'finished.', flush=True)

    def run(self, snapshot_seconds=600):
        if snapshot_seconds < 60:
            raise ValueError('Use at least 60 seconds between recovery ZIPs.')
        self.stop.clear()
        self.snapshot(baseline=True)
        self.snapshot()
        def gpu_one():
            self.run_dataset('davis', 1)
            self.run_dataset('metz', 1)
        executor = ThreadPoolExecutor(max_workers=2)
        tasks = {executor.submit(self.run_dataset, 'kiba', 0), executor.submit(gpu_one)}
        last = time.monotonic()
        last_progress = last
        try:
            while tasks:
                finished, tasks = wait(tasks, timeout=30, return_when=FIRST_COMPLETED)
                for task in finished:
                    task.result()
                if time.monotonic()-last_progress >= 60:
                    progress = {ds: int(self.state(ds)[1]['epoch'])+1 for ds in self.selected}
                    print('Completed epochs:', progress, 'total target:', self.target, flush=True)
                    last_progress = time.monotonic()
                if time.monotonic()-last >= snapshot_seconds:
                    self.snapshot(); last = time.monotonic()
        finally:
            original_error = sys.exc_info()[1]
            self.stop.set()
            executor.shutdown(wait=True)
            try:
                self.snapshot()
            except Exception as backup_error:
                print('Recovery ZIP failed:', backup_error, 'Keep the training_runs folder.', flush=True)
                if original_error is None:
                    raise

    def validation_summary(self):
        result = {}
        for ds in DATASETS:
            _, state = self.state(ds)
            old = self.baseline[ds]['validation_mse']
            best = float(state['best_score'])
            result[ds] = dict(completed_epochs=int(state['epoch'])+1, baseline_best_validation_mse=old,
                             best_validation_mse=best, best_epoch=int(state['best_epoch'])+1,
                             absolute_mse_improvement=old-best, improved=best < old)
        (self.root / 'continuation_validation_summary.json').write_text(json.dumps(result, indent=2))
        print(json.dumps(result, indent=2))
        return result


def validation_curve(folder):
    values, epoch = {}, None
    log = Path(folder) / 'verbose.log'
    if not log.exists():
        return values
    for line in log.read_text(errors='replace').splitlines():
        match = re.search(r'\bEpoch (\d+)\b', line)
        if match:
            epoch = int(match.group(1)) + 1
        match = re.search(r'Validation mse = (\S+)', line)
        if match and epoch is not None:
            try:
                value = float(match.group(1))
                if math.isfinite(value):
                    values[epoch] = value
            except ValueError:
                pass
            epoch = None
    return dict(sorted(values.items()))


In [ ]:
ROOT = prepare_root(ROOT, INPUT, TRAINING_PACKAGE, RESTORE_ZIP)
print('Continuation workspace:', ROOT)


## Check both GPUs and dependencies
Keep Kaggle's installed GPU PyTorch. Package installation runs in a child process, then imports are checked in a fresh process before any training. If dependency resolution fails, the error is printed and training stops; checkpoints are retained.


In [ ]:
import importlib.metadata
probe = subprocess.run([sys.executable, '-c', "import torch,json; assert torch.cuda.is_available(), 'Enable a GPU accelerator'; assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'; [torch.ones((32,32),device='cuda:'+str(i)).matmul(torch.ones((32,32),device='cuda:'+str(i))).sum().item() for i in range(2)]; print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'gpus':[torch.cuda.get_device_name(i) for i in range(2)]}))"], text=True, capture_output=True)
if probe.returncode:
    raise RuntimeError('GPU check failed. Choose GPU T4 x2 in Kaggle.\n' + probe.stderr)
print(probe.stdout)
# Prevent pip from replacing the installed Torch/CUDA stack.
constraints = []
for package in ('torch', 'torchvision', 'torchaudio'):
    try:
        constraints.append(package + '==' + importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        pass
constraint_file = ROOT / 'keep_kaggle_torch.txt'
constraint_file.write_text('\n'.join(constraints)+'\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraint_file),
                '-r', str(ROOT / 'requirements-kaggle.txt')], check=True)
check_code = "import sys, json, importlib.metadata; sys.path.insert(0, sys.argv[1]); import torch, rdkit, tape, chemprop; from chemprop.train.run_training import run_training; print(json.dumps({p:importlib.metadata.version(p) for p in ['torch','numpy','pandas','scipy','scikit-learn','rdkit','tape-proteins','typed-argument-parser','tensorboardX','lifelines']}))"
check = subprocess.run([sys.executable, '-c', check_code, str(ROOT / 'model_source')],
                       text=True, capture_output=True)
if check.returncode:
    raise RuntimeError('Training dependency check failed before training.\n' + check.stderr)
(ROOT / 'kaggle_environment.json').write_text(check.stdout)
print(check.stdout)


## Verify saved models and preserve the restored baseline
Model weights must have the correct shapes and contain only finite numbers. The saved optimizer and random states are restored by the original trainer. The original training loop **does not advance its learning-rate scheduler**; this notebook preserves the saved behavior for a comparable continuation. It is not equivalent to starting a new 60-epoch learning-rate schedule.

`model.pt` holds the best validation model. `resume.pt` holds the latest completed epoch plus optimizer, scheduler, random states and best weights. A periodic ZIP rebuilds `model.pt` from that atomic resume state, avoiding a partly-written best-model file.


In [ ]:
manager = Continuation(ROOT, OUTPUT, TARGET_EPOCHS, DATASETS_TO_TRAIN, BATCH_SIZE)
manager.snapshot(baseline=True)
manager.snapshot()
estimate = manager.estimates()
(ROOT / 'continuation_time_estimate.json').write_text(json.dumps(estimate, indent=2))


## Continue training
GPU 0 runs KIBA; GPU 1 runs Davis then Metz. Each model uses one GPU. Unselected models remain at their restored epoch. A progress message prints about once a minute and detailed logs are under `cpi_resume60/kaggle_logs`.

Training saves **after each completed epoch**. An ordinary interruption attempts a final ZIP. A forced session shutdown can prevent that final save, and unsaved Kaggle working files may disappear. Download/preserve the latest ZIP: it can lag by about 10 minutes plus ZIP-writing time. The original input archive remains unchanged. No automatic backup can guarantee recovery from every platform failure.

The original trainer computes test metrics when a training stage finishes. **Use only validation results to decide whether to continue**; reserve the held-out test results for the final report. The notebook's comparison below uses no test scores.


In [ ]:
if RUN_CONTINUATION:
    manager.run(SNAPSHOT_INTERVAL_SECONDS)
    print('Selected datasets reached their total target. Compare validation below.')
else:
    print('Preflight only. No training was started.')


In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
summary = manager.validation_summary()
report_json = OUTPUT / 'cpi_validation_comparison.json'
report_json.write_text(json.dumps(summary, indent=2))
report_csv = OUTPUT / 'cpi_validation_comparison.csv'
with report_csv.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['dataset']+list(next(iter(summary.values())).keys()))
    writer.writeheader()
    writer.writerows(dict(dataset=ds, **row) for ds, row in summary.items())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, ds in zip(axes, DATASETS):
    curve = validation_curve(manager.folder(ds))
    axis.plot(list(curve), list(curve.values()), label='Validation MSE', linewidth=1.5)
    baseline = manager.baseline[ds]
    axis.axvline(baseline['completed_epochs'], color='gray', linestyle=':', label='Original end')
    axis.axhline(baseline['validation_mse'], color='orange', linestyle='--', label='Original best')
    axis.set(title=ds.upper(), xlabel='Completed epoch', ylabel='Validation MSE (lower is better)')
    axis.grid(alpha=.2)
axes[0].legend(fontsize=8)
fig.tight_layout()
plot = OUTPUT / 'cpi_validation_curves.png'
fig.savefig(plot, dpi=160)
plt.show()
print('If improved=false, keep the existing best model. More epochs are not automatically useful.')
print('Do not call these validation MSEs accuracy percentages or final test results.')
for path in [OUTPUT / 'cpi_training_checkpoints_baseline.zip',
             OUTPUT / f'cpi_training_checkpoints_resume_{TARGET_EPOCHS}.zip', report_json, report_csv, plot]:
    print('Download:', path)
    display(FileLink(str(path)))


## Resume after another interruption / decide the next experiment
Keep the latest `cpi_training_checkpoints_resume_60.zip`. In a fresh Kaggle session, attach it with the original training dataset and set `RESTORE_ZIP` to its exact path. The automatic search will ask for a path if both baseline and continuation ZIPs are attached. Run all cells; it skips completed datasets and continues unfinished ones. The first baseline comparison is carried forward inside the recovery archive.

If validation has continued improving at 60, consider another planned stage with a total target of 150 and the same settings. If it has stalled, more epochs alone are unlikely to help. Consider a separate validation-based tuning experiment, preserving this baseline and leaving test data out of model selection.

The restored-baseline ZIP is the starting state of the current working session. Preserve the original 30-epoch input ZIP separately. Reports/predictions inside an intermediate ZIP can belong to an earlier finished stage: check `epochs_requested` in each training report. `resume.pt` and snapshot metadata are authoritative for completed epochs.

This notebook checks common failure paths but cannot guarantee zero runtime errors or improved generalization. Local CPU checks do not certify a complete Kaggle GPU run. It also does not fix ESMFold downloads or 3D inference errors; affinity training and structure prediction are separate components.

**Known original model limitation:** The original model averages padded neighbor slots, so batch composition can slightly change a score. A verified Davis example changed from 5.933935642 to 5.936890125 when companion molecules changed padding. The site already uses deterministic single-pair inference. Keep the same training batch size and validation settings for continuation; changing aggregation requires a separate retraining and validation experiment.
